# Notebook 05 — Đánh giá mô hình
## Classification Report, Confusion Matrix, ROC-AUC

**Metric chính:**
- Recall(Dropout) — ưu tiên tối đa
- F1-macro — đánh giá cân bằng 3 lớp
- AUC-ROC (one-vs-rest)


In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import warnings
warnings.filterwarnings('ignore')

from src.models.evaluate import (
    full_evaluation, plot_confusion_matrix, plot_roc_curves,
    plot_model_comparison, get_metrics_dict, print_classification_report
)
from src.data.load_data import FEATURE_COLS

plt.rcParams['figure.dpi'] = 120
print('Import thành công ✅')

## 1. Load model và test data

In [ ]:
X_test  = pd.read_csv('../data/processed/X_test.csv').values
y_test  = pd.read_csv('../data/processed/y_test.csv').squeeze().values.astype(int)

best_model = joblib.load('../models/best_model.pkl')

print(f'X_test : {X_test.shape}')
print(f'Model  : {type(best_model).__name__}')

## 2. Đánh giá toàn diện trên Test Set

In [ ]:
metrics = full_evaluation(
    best_model, X_test, y_test,
    model_name=type(best_model).__name__,
    figures_dir='../reports/figures'
)

## 3. Confusion Matrix chi tiết

In [ ]:
y_pred = best_model.predict(X_test)
fig = plot_confusion_matrix(
    y_test, y_pred,
    title=f'Confusion Matrix — {type(best_model).__name__} (Test Set)',
    save_path='../reports/figures/05_confusion_matrix.png'
)
plt.show()

## 4. ROC Curves

In [ ]:
if hasattr(best_model, 'predict_proba'):
    y_prob = best_model.predict_proba(X_test)
    fig = plot_roc_curves(
        y_test, y_prob,
        title=f'ROC Curves — {type(best_model).__name__}',
        save_path='../reports/figures/05_roc_curves.png'
    )
    plt.show()
else:
    print('Mô hình không hỗ trợ predict_proba')

## 5. So sánh tất cả mô hình trên Test Set

In [ ]:
import os
if os.path.exists('../models/model_comparison.csv'):
    df_cmp = pd.read_csv('../models/model_comparison.csv')
    print('Bảng so sánh CV:')
    display(df_cmp.style.highlight_max(
        subset=[c for c in df_cmp.columns if c != 'Model'],
        color='#d4edda'
    ))

    fig = plot_model_comparison(
        df_cmp, metric='f1_macro',
        save_path='../reports/figures/05_model_comparison.png'
    )
    plt.show()

## 6. Tổng kết đánh giá

In [ ]:
print('=== TỔNG KẾT ĐÁNH GIÁ ===')
for k, v in metrics.items():
    status = '✅' if v and v >= 0.7 else '⚠️'
    print(f'  {status} {k:20}: {v}')

print()
if metrics.get('recall_dropout', 0) >= 0.75:
    print('✅ Đạt mục tiêu: Recall(Dropout) ≥ 0.75')
else:
    print('⚠️  Chưa đạt: Recall(Dropout) < 0.75 — cần điều chỉnh threshold')

if metrics.get('f1_macro', 0) >= 0.70:
    print('✅ Đạt mục tiêu: F1-macro ≥ 0.70')
else:
    print('⚠️  Chưa đạt: F1-macro < 0.70')